# Splane Mouse Hypothalamus (1/2): Identify uniform spatial domain

## Data preprocessing

In [1]:
from SPACEL.setting import set_environ_seed
seed = 2024
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib
import anndata
import glob
import os
from natsort import natsorted

Setting environment seed: 2024


Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [ ]:
# Path to the directory
data_dir = './mHemi_brain'

# Find all .h5ad files
h5ad_files = natsorted(glob.glob(os.path.join(data_dir, '*.h5ad')))
# h5ad_files
adata_list = []
# Read all .h5ad files into a list of AnnData objects
for f in h5ad_files:
    ad = sc.read(f)
    ad.obs['neurotransmitter'] = ad.obs['neurotransmitter'].cat.add_categories("NaN")
    ad.obs['original_clusters'] = ad.obs['neurotransmitter'].fillna("NaN").astype(str)
#     Splane.utils.add_cell_type_composition(ad, celltype_anno=ad.obs['class'])
    adata_list.append(ad)

In [3]:
ad.obs['class'].unique().to_list

<bound method Categorical.to_list of ['28 CB GABA', '34 Immune', '33 Vascular', '30 Astro-Epen', '29 CB Glut', ..., '24 MY Glut', '13 CNU-HYa Glut', '23 P Glut', '09 CNU-LGE GABA', '02 NP-CT-L6b Glut']
Length: 14
Categories (14, object): ['02 NP-CT-L6b Glut', '09 CNU-LGE GABA', '12 HY GABA', '13 CNU-HYa Glut', ..., '30 Astro-Epen', '31 OPC-Oligo', '33 Vascular', '34 Immune']>

In [4]:
adata_list[2]

AnnData object with n_obs × n_vars = 8040 × 1122
    obs: 'brain_section_label', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'z', 'subclass_confidence_score', 'cluster_confidence_score', 'high_quality_transfer', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'original_clusters'
    var: 'gene_symbol', 'name', 'mapped_ncbi_identifier'
    obsm: 'ccf_spatial', 'spatial'

In [5]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['class'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
# all_cell_types

In [6]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['class'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

In [7]:
adata_list[1].uns['celltypes']

array(['01 IT-ET Glut', '02 NP-CT-L6b Glut', '03 OB-CR Glut',
       '04 DG-IMN Glut', '05 OB-IMN GABA', '06 CTX-CGE GABA',
       '07 CTX-MGE GABA', '08 CNU-MGE GABA', '09 CNU-LGE GABA',
       '10 LSX GABA', '11 CNU-HYa GABA', '12 HY GABA', '13 CNU-HYa Glut',
       '14 HY Glut', '15 HY Gnrh1 Glut', '16 HY MM Glut', '17 MH-LH Glut',
       '18 TH Glut', '19 MB Glut', '20 MB GABA', '21 MB Dopa',
       '22 MB-HB Sero', '23 P Glut', '24 MY Glut', '25 Pineal Glut',
       '26 P GABA', '27 MY GABA', '28 CB GABA', '29 CB Glut',
       '30 Astro-Epen', '31 OPC-Oligo', '32 OEC', '33 Vascular',
       '34 Immune'], dtype=object)

## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [ ]:
for i in range(3):
    if i == 2:
        data = adata_list[i*40:]
    else:
        data = adata_list[i*40:(i+1)*40]

    splane_model = Splane.init_model(data, n_clusters=10,use_gpu=True,n_neighbors=25, gnn_dropout=0.5, seed=seed)
    splane_model.train(d_l=0.2)
    splane_model.identify_spatial_domain()
    
    sc.concat(data).write(f'./{seed}/mHemi_Brain_Splane{i}.h5ad')    

Setting global seed: 2024
Calculating cell type weights...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/utils.py:76: RuntimeWarning: invalid value encountered in double_scalars
  return len(x) / W * inum / z2ss


Generating GNN inputs...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:11: RuntimeWarning: divide by zero encountered in power
  d = sp.diags(np.power(np.array(adj.sum(1)), -1).flatten(), 0)


Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


... (13 outputs left out)

/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:11: RuntimeWarning: divide by zero encountered in power
  d = sp.diags(np.power(np.array(adj.sum(1)), -1).flatten(), 0)


Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...
Setting the weight of similarity loss to 9.669


The best epoch 80 total loss=-10.848 g loss=-10.075 d loss=3.865 d acc=0.061 simi loss=-0.996 db loss=0.594:  45%|████▌     | 135/300 [12:21<15:06,  5.49s/it]

Stop trainning because of loss convergence


In [9]:
adata_list[0]

AnnData object with n_obs × n_vars = 8027 × 1122
    obs: 'brain_section_label', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'z', 'subclass_confidence_score', 'cluster_confidence_score', 'high_quality_transfer', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'original_clusters', '01 IT-ET Glut', '02 NP-CT-L6b Glut', '03 OB-CR Glut', '04 DG-IMN Glut', '05 OB-IMN GABA', '06 CTX-CGE GABA', '07 CTX-MGE GABA', '08 CNU-MGE GABA', '09 CNU-LGE GABA', '10 LSX GABA', '11 CNU-HYa GABA', '12 HY GABA', '13 CNU-HYa Glut', '14 HY Glut', '15 HY Gnrh1 Glut', '16 HY MM Glut', '17 MH-LH Glut', '18 TH Glut', '19 MB Glut', '20 MB GABA', '21 MB Dopa', '22 MB-HB Sero', '23 P Glut', '24 MY Glut', '25 Pineal Glut', '26 P GABA', '27 MY GABA', '28 CB GABA', '29 CB Glut', '30 Astro-Epen', '31 OPC-Oligo', '32 OEC', '33 Vascular', '34 Immune', 'spatial_domain'
 